In [5]:
import json
import os
import numpy as np

In [ ]:
def load_json(path):
    with open(path, "r") as f:
        return json.load(f)

def extract_snr_dict(data):
    snr_dict = {}
    for timestamp, antenna_data in data.items():
        pulses = antenna_data.get("Antenna 6", {}).get("pulse_data", [])
        for pulse in pulses:
            start = pulse["start"]
            end = pulse["end"]
            sats_present = pulse.get("sats_present", {})

            for satID, detections in sats_present.items():
                max_snr = max(det[2] for det in detections)
                key = (timestamp, start, end, satID)
                snr_dict[key] = max_snr
    return snr_dict

def compare_snrs(snr_dict1, snr_dict2):
    all_keys = set(snr_dict1.keys()) | set(snr_dict2.keys())
    comparison = []

    for key in sorted(all_keys):
        snr1 = snr_dict1.get(key, None)
        snr2 = snr_dict2.get(key, None)

        comparison.append({
            "timestamp": key[0],
            "pulse_start": key[1],
            "pulse_end": key[2],
            "satID": key[3],
            "SNR_file1": snr1,
            "SNR_file2": snr2,
            "diff": snr1 - snr2 if snr1 is not None and snr2 is not None else None
        })
    return comparison

def get_diff_list(comparison, skip_missing = True):
    diff_list = []
    for entry in comparison:
        diff = entry['diff']
        if skip_missing and diff is None:
            continue
        diff_list.append(diff)
    return diff_list

def print_comparison_table(comparison):
    print(f" {'Start':>6} {'End':>6} {'SatID':>8} {'SNR1':>10} {'SNR2':>10} {'ΔSNR':>10}")
    print("-"*64)
    for item in comparison:
        print(f"{item['pulse_start']:>6} {item['pulse_end']:>6} {item['satID']:>8} "
              f"{item['SNR_file1'] if item['SNR_file1'] is not None else '—':>10} "
              f"{item['SNR_file2'] if item['SNR_file2'] is not None else '—':>10} "
              f"{item['diff'] if item['diff'] is not None else '—':>10}")
        
def print_no_sat_times(data):
    time_set = set()
    for timestamp, all_data in data.items():
        for antname, antenna_data in all_data.items():
            for pulse in antenna_data['pulse_data']:
                start = pulse["start"]
                end = pulse["end"]
                time_set.add((start, end))
        
    times = list(time_set)
    sorted_times = sorted(times, key=lambda x : x[0])

    gaps = []
    lengths = []
    for i in range(1, len(sorted_times)):
        start = sorted_times[i-1][1]
        end = sorted_times[i][0]
        gaps.append((start, end))
        lengths.append(end-start)

    #print('all sat gaps:')
    #for gap in gaps:
    #    print(gap)
    print('mean seconds with no sat risen', int(np.mean(lengths)))
    print('in minutes:', int(np.mean(lengths)/60))
    print('longest in minutes:', int(max(lengths)/60))

In [34]:

# === USAGE ===
dir = '/scratch/thomasb'

config_uncorr = os.path.join(dir, "pulsedata_1753132820_len_67200_1760024247.5361912.json")
config_corr = os.path.join(dir, "pulsedata_1753132820_len_67200_1760452124.3504817.json")

data_uncorr = load_json(config_uncorr)
data_corr = load_json(config_corr)

snr_dict_uncorr = extract_snr_dict(data_uncorr)
snr_dict_corr = extract_snr_dict(data_corr)

comparison = compare_snrs(snr_dict_uncorr, snr_dict_corr)
print_comparison_table(comparison)
diffs = get_diff_list(comparison)
print(np.mean(diffs))


  Start    End    SatID       SNR1       SNR2       ΔSNR
----------------------------------------------------------------
   525    885    25338         36         36          0
   885   1065    25338        307        307          0
  1425   1985    33591         75         76         -1
  4860   5170    57166         23         23          0
  5170   5395    57166         89         88          1
  6925   7070    25338        335        333          2
 12595  12965    25338        188        190         -2
 12965  13075    25338        155        152          3
 13515  14080    33591         35         35          0
 18720  19020    25338        127        125          2
 19020  19085    25338         63         64         -1
 29190  29450    57166         40         40          0
 31995  32260    33591        164        164          0
 34720  35030    59051        269        271         -2
 40740  41200    59051        117        117          0
 46770  47295    59051        224     

In [35]:
print_no_sat_times(data_uncorr)

all sat gaps:
(885, 885)
(1065, 1425)
(1985, 3980)
(4320, 4860)
(5170, 5170)
(5395, 6535)
(6925, 6925)
(7070, 7465)
(8025, 10865)
(10935, 10935)
(11170, 11170)
(11405, 12595)
(12965, 12965)
(13075, 13515)
(14080, 16905)
(17420, 18720)
(19020, 19020)
(19085, 23005)
(23440, 25025)
(25085, 25760)
(26205, 29190)
(29450, 31995)
(32260, 34720)
(35030, 40740)
(41200, 46770)
(47295, 49385)
(49665, 52790)
(53335, 53860)
(53930, 55385)
(55825, 56800)
(56985, 58805)
(59340, 59820)
(60135, 60135)
(60190, 61395)
(61720, 61720)
(61910, 62840)
(63250, 64805)
(65345, 65840)
(66130, 66130)
mean seconds with no sat risen 1362
in minutes: 22
longest in minutes: 95
